# Publish an editable deployment-only SageMaker Project template

This notebook turns the AWS deployment toolchain snapshot into an **organization-owned S3 project template**. It packages [`deployment-seed-code/`](deployment-seed-code/) into a ZIP, uploads that ZIP and [`cfn/deployment_only_project_template.yml`](cfn/deployment_only_project_template.yml) to your S3 bucket, tags the template for SageMaker Studio visibility, and can create a SageMaker Project directly with `TemplateProviders/CfnTemplateProvider`.

There is no model-build pipeline. The resulting Project creates only the deployment path: Model Registry and source events → CodePipeline → staging endpoint → test → manual approval → production endpoint.

> **Support caveat:** this minimal derivative retains CodeCommit because it matches the copied AWS stack and this workshop account has existing access. AWS labels the original product not supported, and CodeCommit is unavailable to new customers. Use the newer CodeConnections-based `deploy-codepipeline.yaml` sample for a new production platform.

> **Cost warning:** publishing two small S3 objects is inexpensive. Creating the Project provisions CodeCommit, CodeBuild, CodePipeline, EventBridge, CloudFormation, and S3 resources. A deployment pipeline run creates a billable staging endpoint and, after approval, a production endpoint. All mutating actions are disabled by default.

## 1. Runtime and AWS context

Run this notebook from the repository root with the workshop's Python 3.12 kernel. It reuses `%store` values when available but does not depend on the model-build notebook.

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}"
)

try:
    get_ipython().run_line_magic("store", "-r")
except Exception:
    pass

In [ ]:
import hashlib
import json
import re
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.parse import quote

import boto3
import yaml
from botocore.exceptions import ClientError

session = boto3.Session()
region = globals().get("region") or session.region_name
if not region:
    raise RuntimeError("No AWS Region is configured.")

sts = session.client("sts", region_name=region)
account_id = sts.get_caller_identity()["Account"]
s3 = session.client("s3", region_name=region)
sm = session.client("sagemaker", region_name=region)
cfn = session.client("cloudformation", region_name=region)
iam = session.client("iam", region_name=region)

default_bucket = globals().get("bucket_name") or f"sagemaker-{region}-{account_id}"
print(f"Region:  {region}")
print(f"Account: {account_id}")
print(f"Default publication bucket: {default_bucket}")

## 2. Settings and safety controls

The three flags separate publication, optional Studio UI discovery, and Project creation. Programmatic Project creation does not require domain tagging or bucket CORS.

In [ ]:
SEED_DIRECTORY = Path("deployment-seed-code")
PROJECT_TEMPLATE = Path("cfn/deployment_only_project_template.yml")
BUILD_DIRECTORY = Path(".build/deployment-project-template")

PUBLICATION_BUCKET = default_bucket
PUBLICATION_PREFIX = "project-templates/deployment-only"
SEED_OBJECT_KEY = f"{PUBLICATION_PREFIX}/seed/deployment-seed-code.zip"
TEMPLATE_OBJECT_KEY = f"{PUBLICATION_PREFIX}/templates/deployment-only-codecommit.yml"

PUBLISH_ARTIFACTS = False       # creates/updates two S3 objects
CONFIGURE_STUDIO_DISCOVERY = False  # updates bucket CORS and a SageMaker domain tag
CREATE_PROJECT = False          # creates the CI/CD toolchain; later runs can create endpoints

DOMAIN_ID = globals().get("domain_id", "")
date_suffix = time.strftime("%d%m%y", time.gmtime())
PROJECT_NAME = f"custom-deploy-{date_suffix}"
MODEL_PACKAGE_GROUP_ARN = ""  # optional in preview; required to create a Project
LAUNCH_ROLE_NAME = "AmazonSageMakerProjectsLaunchRole"
USE_ROLE_NAME = "AmazonSageMakerServiceCatalogProductsUseRole"

print(f"Seed target:     s3://{PUBLICATION_BUCKET}/{SEED_OBJECT_KEY}")
print(f"Template target: s3://{PUBLICATION_BUCKET}/{TEMPLATE_OBJECT_KEY}")

## 3. Validate and package the editable seed repository

Files are stored at the ZIP root because `AWS::CodeCommit::Repository.Code` expands that root into the new repository. Fixed timestamps and permissions make the archive reproducible: unchanged source produces the same SHA-256. `ORIGIN.md` is included so each generated repository retains provenance.

In [ ]:
required_seed_files = {
    Path("build.py"),
    Path("buildspec.yml"),
    Path("endpoint-config-template.yml"),
    Path("staging-config.json"),
    Path("prod-config.json"),
    Path("test/buildspec.yml"),
    Path("test/test.py"),
}

if not SEED_DIRECTORY.is_dir():
    raise FileNotFoundError(SEED_DIRECTORY)
if not PROJECT_TEMPLATE.is_file():
    raise FileNotFoundError(PROJECT_TEMPLATE)

seed_files = sorted(
    path for path in SEED_DIRECTORY.rglob("*")
    if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc"
)
relative_seed_files = {path.relative_to(SEED_DIRECTORY) for path in seed_files}
missing = required_seed_files - relative_seed_files
if missing:
    raise FileNotFoundError(f"Missing required seed files: {sorted(map(str, missing))}")

compile((SEED_DIRECTORY / "build.py").read_text(), "build.py", "exec")
compile((SEED_DIRECTORY / "test/test.py").read_text(), "test/test.py", "exec")
json.loads((SEED_DIRECTORY / "staging-config.json").read_text())
json.loads((SEED_DIRECTORY / "prod-config.json").read_text())
yaml.safe_load((SEED_DIRECTORY / "buildspec.yml").read_text())
yaml.safe_load((SEED_DIRECTORY / "test/buildspec.yml").read_text())
yaml.compose((SEED_DIRECTORY / "endpoint-config-template.yml").read_text())

print(f"Validated {len(seed_files)} seed files:")
for path in seed_files:
    print(f"  {path.relative_to(SEED_DIRECTORY)}")

In [ ]:
BUILD_DIRECTORY.mkdir(parents=True, exist_ok=True)
seed_zip = BUILD_DIRECTORY / "deployment-seed-code.zip"

with zipfile.ZipFile(seed_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
    for source_path in seed_files:
        relative_path = source_path.relative_to(SEED_DIRECTORY).as_posix()
        info = zipfile.ZipInfo(relative_path, date_time=(2023, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        archive.writestr(info, source_path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)

with zipfile.ZipFile(seed_zip) as archive:
    archived_files = set(archive.namelist())
    expected_files = {path.relative_to(SEED_DIRECTORY).as_posix() for path in seed_files}
    assert archived_files == expected_files
    assert all(not name.startswith("deployment-seed-code/") for name in archived_files)

seed_sha256 = hashlib.sha256(seed_zip.read_bytes()).hexdigest()
print(f"Built {seed_zip} ({seed_zip.stat().st_size:,} bytes)")
print(f"SHA-256: {seed_sha256}")

## 4. Validate the deployment-only project template

The AWS snapshot remains unchanged under `cfn/endpoint_deployment_toolchain_template.yml`. This derivative adds only `DeploymentSeedCodeBucket` and `DeploymentSeedCodeKey`, replacing the hard-coded AWS seed location with references to your uploaded ZIP. SageMaker injects `SageMakerProjectName` and `SageMakerProjectId`; callers provide the model package group and seed location.

In [ ]:
template_bytes = PROJECT_TEMPLATE.read_bytes()
template = yaml.safe_load(template_bytes)
expected_parameters = {
    "SageMakerProjectName",
    "SageMakerProjectId",
    "SourceModelPackageGroupName",
    "DeploymentSeedCodeBucket",
    "DeploymentSeedCodeKey",
}
assert set(template["Parameters"]) == expected_parameters

repository = template["Resources"]["ModelDeployCodeCommitRepository"]
seed_source = repository["Properties"]["Code"]["S3"]
assert seed_source["Bucket"] == {"Ref": "DeploymentSeedCodeBucket"}
assert seed_source["Key"] == {"Ref": "DeploymentSeedCodeKey"}

resource_types = Counter(resource["Type"] for resource in template["Resources"].values())
assert resource_types["AWS::CodeCommit::Repository"] == 1
assert resource_types["AWS::CodePipeline::Pipeline"] == 1
assert resource_types["AWS::CodeBuild::Project"] == 2
assert resource_types["AWS::Events::Rule"] == 2

template_sha256 = hashlib.sha256(template_bytes).hexdigest()
pipeline = template["Resources"]["ModelDeployPipeline"]
stages = [stage["Name"] for stage in pipeline["Properties"]["Stages"]]
print(f"Template SHA-256: {template_sha256}")
print(f"Parameters: {sorted(expected_parameters)}")
print(f"Pipeline: {' -> '.join(stages)}")

## 5. Check prerequisite roles

The derivative intentionally preserves the roles referenced by the copied workshop stack. The launch role provisions the template; the existing Service Catalog products use role is assumed by CodePipeline, CodeBuild, EventBridge, CloudFormation, and the deployed SageMaker model. The launch role also needs `s3:GetObject` for both published objects.

In [ ]:
role_arns = {}
missing_roles = []
for role_name in [LAUNCH_ROLE_NAME, USE_ROLE_NAME]:
    try:
        role_arns[role_name] = iam.get_role(RoleName=role_name)["Role"]["Arn"]
        print(f"Found {role_name}: {role_arns[role_name]}")
    except iam.exceptions.NoSuchEntityException:
        missing_roles.append(role_name)
        print(f"MISSING: {role_name}")

if CREATE_PROJECT and missing_roles:
    raise RuntimeError(f"Cannot create the Project; missing roles: {missing_roles}")

## 6. Publish the seed ZIP and template to S3

Set `PUBLISH_ARTIFACTS = True` in the settings cell, then rerun from there. Existing objects at these keys are versioned only if bucket versioning is enabled; use a different prefix when you need immutable releases. The template object receives `sagemaker:studio-visibility=true`.

In [ ]:
PUBLISH_ARTIFACTS = True

In [ ]:
def ensure_bucket(bucket_name):
    try:
        s3.head_bucket(Bucket=bucket_name)
        return False
    except ClientError as error:
        status = error.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if status not in {404}:
            raise
    request = {"Bucket": bucket_name}
    if region != "us-east-1":
        request["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**request)
    return True


if not PUBLISH_ARTIFACTS:
    print("PREVIEW ONLY — set PUBLISH_ARTIFACTS = True to upload the artifacts.")
else:
    created = ensure_bucket(PUBLICATION_BUCKET)
    print(f"{'Created' if created else 'Using'} bucket {PUBLICATION_BUCKET}")

    cfn.validate_template(TemplateBody=template_bytes.decode("utf-8"))
    s3.upload_file(
        str(seed_zip), PUBLICATION_BUCKET, SEED_OBJECT_KEY,
        ExtraArgs={"ContentType": "application/zip", "Metadata": {"sha256": seed_sha256}},
    )
    s3.upload_file(
        str(PROJECT_TEMPLATE), PUBLICATION_BUCKET, TEMPLATE_OBJECT_KEY,
        ExtraArgs={"ContentType": "application/x-yaml", "Metadata": {"sha256": template_sha256}},
    )
    s3.put_object_tagging(
        Bucket=PUBLICATION_BUCKET,
        Key=TEMPLATE_OBJECT_KEY,
        Tagging={"TagSet": [{"Key": "sagemaker:studio-visibility", "Value": "true"}]},
    )

    for key, expected_sha in [(SEED_OBJECT_KEY, seed_sha256), (TEMPLATE_OBJECT_KEY, template_sha256)]:
        uploaded = s3.head_object(Bucket=PUBLICATION_BUCKET, Key=key)
        assert uploaded["Metadata"]["sha256"] == expected_sha
        print(f"Published s3://{PUBLICATION_BUCKET}/{key} ({uploaded['ContentLength']:,} bytes)")

## 7. Optionally expose the template in the Studio UI

This is not needed for the programmatic launch below. Studio discovery requires the domain tag `sagemaker:projectS3TemplatesLocation` and bucket CORS. The cell preserves existing CORS rules and adds the required read rule if absent. Set `DOMAIN_ID` explicitly if it was not restored from notebook 00.

In [ ]:
CONFIGURE_STUDIO_DISCOVERY = True

In [ ]:
studio_cors_rule = {
    "AllowedHeaders": ["*"],
    "AllowedMethods": ["GET"],
    "AllowedOrigins": ["*"],
    "ExposeHeaders": ["ETag", "x-amz-version-id"],
    "MaxAgeSeconds": 3000,
}

if not CONFIGURE_STUDIO_DISCOVERY:
    print("Skipped — programmatic Project creation does not require Studio discovery.")
else:
    if not PUBLISH_ARTIFACTS:
        raise RuntimeError("Publish the template before configuring Studio discovery.")
    if not DOMAIN_ID:
        domains = sm.list_domains(MaxResults=100)["Domains"]
        if len(domains) != 1:
            raise RuntimeError(f"Set DOMAIN_ID explicitly; found {len(domains)} domains.")
        DOMAIN_ID = domains[0]["DomainId"]

    try:
        cors_rules = s3.get_bucket_cors(Bucket=PUBLICATION_BUCKET)["CORSRules"]
    except ClientError as error:
        if error.response["Error"]["Code"] != "NoSuchCORSConfiguration":
            raise
        cors_rules = []
    if studio_cors_rule not in cors_rules:
        cors_rules.append(studio_cors_rule)
        s3.put_bucket_cors(
            Bucket=PUBLICATION_BUCKET, CORSConfiguration={"CORSRules": cors_rules}
        )

    domain = sm.describe_domain(DomainId=DOMAIN_ID)
    template_location = f"s3://{PUBLICATION_BUCKET}/{PUBLICATION_PREFIX}/templates/"
    sm.add_tags(
        ResourceArn=domain["DomainArn"],
        Tags=[{"Key": "sagemaker:projectS3TemplatesLocation", "Value": template_location}],
    )
    print(f"Configured domain {DOMAIN_ID} to discover {template_location}")

## 8. Select the Model Package Group

Supply a group ARN, not a package-version ARN. If earlier notebooks stored a group name, this cell resolves it automatically. The Project template receives the final group name and its EventBridge rule watches that group.

In [ ]:
from governance.approval_gate import resolve_model_package_group

MODEL_PACKAGE_GROUP_ARN = ""  # e.g. arn:aws:sagemaker:us-west-2:123456789012:model-package-group/my-group

if not MODEL_PACKAGE_GROUP_ARN:
    workshop_group_name = resolve_model_package_group(
        sm, registered_model_name, prefix="from-idea-to-prod-model-"
    )
    MODEL_PACKAGE_GROUP_ARN = sm.describe_model_package_group(
        ModelPackageGroupName=workshop_group_name
    )["ModelPackageGroupArn"]
    print("No override supplied; using the workshop group.")

print(f"Model Package Group ARN: {MODEL_PACKAGE_GROUP_ARN}")

model_package_group_name = None
if MODEL_PACKAGE_GROUP_ARN:
    match = re.fullmatch(
        r"arn:(?P<partition>aws(?:-[a-z]+)*):sagemaker:(?P<region>[^:]+):"
        r"(?P<account>\d{12}):model-package-group/(?P<name>[A-Za-z0-9](?:-*[A-Za-z0-9]){0,62})",
        MODEL_PACKAGE_GROUP_ARN,
    )
    if not match:
        raise ValueError("MODEL_PACKAGE_GROUP_ARN must be a Model Package Group ARN.")
    if match["region"] != region or match["account"] != account_id:
        raise ValueError("The Model Package Group must be in this account and Region.")
    model_package_group_name = match["name"]
    try:
        described = sm.describe_model_package_group(ModelPackageGroupName=model_package_group_name)
    except ClientError:
        if not group_restored_from_store or CREATE_PROJECT:
            raise
        print(f"Ignoring stale stored Model Package Group: {MODEL_PACKAGE_GROUP_ARN}")
        MODEL_PACKAGE_GROUP_ARN = ""
        model_package_group_name = None
    else:
        assert described["ModelPackageGroupArn"] == MODEL_PACKAGE_GROUP_ARN
        print(f"Deployment source: {MODEL_PACKAGE_GROUP_ARN}")
elif CREATE_PROJECT:
    raise RuntimeError("Set MODEL_PACKAGE_GROUP_ARN before creating the Project.")
else:
    print("No group selected — acceptable for artifact publication and launch preview only.")

## 9. Create the deployment-only SageMaker Project

This uses the S3 template directly—there is no custom Service Catalog product to create or discover. SageMaker supplies the project name/ID parameters. The launch call supplies only the Model Package Group and your seed object coordinates. Set both `PUBLISH_ARTIFACTS = True` and `CREATE_PROJECT = True` when ready.

In [ ]:
PUBLISH_ARTIFACTS = True
CREATE_PROJECT = True

In [ ]:
if not re.fullmatch(r"[A-Za-z](?:-*[A-Za-z0-9]){0,31}", PROJECT_NAME):
    raise ValueError("PROJECT_NAME must start with a letter and contain at most 32 letters, digits, or hyphens.")

template_url = (
    f"https://{PUBLICATION_BUCKET}.s3.{region}.amazonaws.com/"
    f"{quote(TEMPLATE_OBJECT_KEY, safe='/')}"
)
project_details = None
project_created_here = False

existing = [
    item for item in sm.list_projects(NameContains=PROJECT_NAME, MaxResults=100)["ProjectSummaryList"]
    if item["ProjectName"] == PROJECT_NAME
]
if existing:
    project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    print(f"Project already exists: {PROJECT_NAME} ({project_details['ProjectStatus']})")
elif not CREATE_PROJECT:
    print("PREVIEW ONLY — no Project created.")
    print(f"Template URL: {template_url}")
    print(f"SourceModelPackageGroupName: {model_package_group_name or '<set MODEL_PACKAGE_GROUP_ARN>'}")
    print(f"DeploymentSeedCodeBucket: {PUBLICATION_BUCKET}")
    print(f"DeploymentSeedCodeKey: {SEED_OBJECT_KEY}")
else:
    if not PUBLISH_ARTIFACTS:
        raise RuntimeError("Set PUBLISH_ARTIFACTS = True and publish before creating the Project.")
    if missing_roles:
        raise RuntimeError(f"Missing required roles: {missing_roles}")
    response = sm.create_project(
        ProjectName=PROJECT_NAME,
        ProjectDescription=(
            "Deployment-only CI/CD for approved SageMaker Model Packages, published from an S3 custom template."
        ),
        TemplateProviders=[{
            "CfnTemplateProvider": {
                "TemplateName": "DeploymentOnlyCodeCommit",
                "TemplateURL": template_url,
                "RoleARN": role_arns[LAUNCH_ROLE_NAME],
                "Parameters": [
                    {"Key": "SourceModelPackageGroupName", "Value": model_package_group_name},
                    {"Key": "DeploymentSeedCodeBucket", "Value": PUBLICATION_BUCKET},
                    {"Key": "DeploymentSeedCodeKey", "Value": SEED_OBJECT_KEY},
                ],
            }
        }],
        Tags=[
            {"Key": "workshop", "Value": "from-idea-to-production"},
            {"Key": "purpose", "Value": "model-deployment"},
        ],
    )
    project_created_here = True
    project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    print(f"Creating {response['ProjectArn']}")

In [ ]:
if project_details is None:
    print("Skipped — enable CREATE_PROJECT when ready.")
elif not project_created_here:
    print(f"Existing Project status: {project_details['ProjectStatus']}")
else:
    terminal = {"CreateCompleted", "CreateFailed", "CreateCompletedWithErrors"}
    deadline = time.time() + 900
    while project_details["ProjectStatus"] not in terminal:
        if time.time() > deadline:
            raise TimeoutError(f"Project creation did not finish: {project_details['ProjectStatus']}")
        print(f"  {time.strftime('%H:%M:%S')} | {project_details['ProjectStatus']}")
        time.sleep(20)
        project_details = sm.describe_project(ProjectName=PROJECT_NAME)
    if project_details["ProjectStatus"] != "CreateCompleted":
        raise RuntimeError(
            f"Project creation failed: {project_details['ProjectStatus']} — "
            f"{project_details.get('ProjectStatusMessage', '')}"
        )
    print(f"Project ready: {PROJECT_NAME} / {project_details['ProjectId']}")

## What to adapt next

The generated CodeCommit repository is initialized from your local `deployment-seed-code/` snapshot. Make changes locally, republish the ZIP, and create a **new** Project to seed a new repository; republishing does not overwrite repositories already created. For an existing Project, edit and push its generated CodeCommit repository.

Before production use:

1. Replace the placeholder in `test/test.py` with a real inference request and assertions.
2. Review instance types, counts, and capture percentages in both stage config files.
3. Decide whether `build.py` should enforce the workshop's `production/active` lifecycle in addition to `Approved`.
4. Move to CodeConnections and your organization’s supported Git provider when this is no longer a workshop-only deployment.
5. Delete staging and production endpoints when finished; deleting the Project does not automatically remove retained S3 buckets.